
# GenAI Evaluation Framework on Gemini Enterprise Agent Platform
### GCP Training Program — Module 7 (part of the 3.5-Hour MLOps block)

**Problem statement:** a support team wants to auto-draft replies to short customer messages using
Gemini. Before shipping this to production, they need a **repeatable, quantitative way** to answer
three questions:

1. **Is a given model/prompt actually good enough?** — scored against fixed criteria, not eyeballed.
2. **Is a cheaper model good enough to replace a more expensive one?** — a **model migration**
   decision, backed by evidence, not guesswork.
3. **Did a prompt change make things better or worse?** — a **prompt regression test**, run the
   same way every time a prompt is edited.

This is exactly the same motivation as unit tests in software engineering — except the "correct
output" for a language model is a matter of degree and judgment, not exact-match pass/fail. The
**Gen AI evaluation service** is Agent Platform's structured way to make that judgment
**consistent, scalable, and trackable over time**, instead of a human skimming a handful of
responses and guessing.

**The dataset:** 8 small, realistic customer support messages, each already paired with a
**reference reply** (what a good human agent would write) — small enough to read end-to-end in
class, large enough to show every technique below meaningfully.

## ⚠️ Terminology & SDK note
This notebook uses `vertexai.Client().evals` — Agent Platform's **current, recommended** interface
for the Gen AI evaluation service (as opposed to the older `vertexai.evaluation.EvalTask` class,
which still works but is the legacy interface). Console navigation is under
**Agent Platform > Models > Evaluation**.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs against
your own project.


## 1. Authenticate & Install

In [1]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")


Authenticated in Colab.


In [3]:

!pip install --upgrade --quiet "google-cloud-aiplatform[evaluation]" "pandas==2.2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 7.9 MB/s eta 0:00:00


In [4]:

import IPython
IPython.Application.instance().kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

## 2. Configure Your Project & Region

In [1]:

PROJECT_ID = input("Enter your GCP Project ID: ").strip()
REGION = input("Enter your Agent Platform region [default: us-central1]: ").strip() or "us-central1"

!gcloud config set project {PROJECT_ID}
!gcloud services enable aiplatform.googleapis.com --project {PROJECT_ID}

import vertexai
client = vertexai.Client(project=PROJECT_ID, location=REGION)

print("Project:", PROJECT_ID)
print("Region:", REGION)


Enter your GCP Project ID: cs-poc-ytkjb6nscjlihtluxkcmuoq
Enter your Agent Platform region [default: us-central1]: 
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey

Project: cs-poc-ytkjb6nscjlihtluxkcmuoq
Region: us-central1



## 3. The Evaluation Dataset

**What this is:** 8 short customer messages, each with a `reference` — the reply a good human
agent would send. This reference is what makes both **computation-based metrics** (Section 4, which
compare wording directly) and some **model-based metrics** (Section 5, which use an LLM to judge
quality) possible — without ground truth to compare against, evaluation would only ever be a matter
of unverified opinion.


In [2]:

import pandas as pd

eval_data = pd.DataFrame({
    "prompt": [
        "My order arrived three days late and the box was crushed, but the product inside looks okay. I'm annoyed but I don't necessarily want a refund, just some acknowledgement.",
        "I was charged twice for the same order. Please fix this immediately, I want my money back.",
        "The size chart on your website said Large would fit me but it's way too small. Can I exchange it for a Extra Large?",
        "Your app keeps crashing every time I try to check out. This is really frustrating.",
        "I love the product, just wondering if you ever restock the blue color?",
        "It's been 2 weeks and my order still says 'processing'. What's going on?",
        "Can I cancel my subscription? I don't think I'll be using it anymore.",
        "The instructions in the manual don't match the actual product. Which version is correct?",
    ],
    "reference": [
        "I'm really sorry your order arrived late and damaged — I completely understand the frustration. I've made a note of this on your account; please let us know if the product itself has any issues and we'll make it right.",
        "I'm sorry for the duplicate charge — that's on us. I've refunded the extra charge to your original payment method; you should see it within 3-5 business days.",
        "Thanks for letting us know! I've started an exchange for a Large to an Extra Large — you'll get a prepaid return label by email, and the new size ships as soon as we receive the original.",
        "I'm sorry the app is crashing at checkout — that's a serious issue and I want to get you unblocked. Could you tell me your device type and app version so we can look into it right away?",
        "So glad you're loving it! Blue is currently out of stock, but I've added your email to the restock notification list so you'll hear the moment it's back.",
        "I'm sorry for the delay — let me look into your order status right now and get you a real update rather than leaving you waiting.",
        "Of course — I've submitted your cancellation request, effective at the end of your current billing period. You won't be charged again after that.",
        "Thanks for flagging that — you're right that there's a mismatch. The product itself is correct; we'll get the manual updated. In the meantime, here's the correct setup steps.",
    ],
})

print(f"{len(eval_data)} evaluation examples loaded.")
eval_data.head(3)


8 evaluation examples loaded.


,prompt,reference
0,My order arrived three days late and the box w...,I'm really sorry your order arrived late and d...
1,I was charged twice for the same order. Please...,I'm sorry for the duplicate charge — that's on...
2,The size chart on your website said Large woul...,Thanks for letting us know! I've started an ex...



## 4. Step 1 — Generate Responses, Then Score With Computation-Based Metrics

### 4.1 Generate Responses (`run_inference`)
**What this does:** sends every prompt in the dataset to a chosen model and collects the responses
— the first of the evaluation service's **two-step process**: generate, then separately evaluate.
Keeping these steps separate is what lets Section 6 reuse the same generated responses against
multiple different metrics without regenerating them each time.


In [3]:

eval_dataset_v1 = client.evals.run_inference(
    model="gemini-2.5-flash",
    src=eval_data,
)
eval_dataset_v1.show()


17:46:45 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
17:46:45 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
Gemini Inference: 100%|██████████| 8/8 [00:17<00:00,  2.22s/it]



### 4.2 Computation-Based Metrics — ROUGE and BLEU
**What these measure:** pure **word-overlap** against the `reference` column — no model judgment
involved, fast and fully deterministic. `rouge_1` measures overlapping single words; `bleu` measures
overlapping short phrases (n-grams). **Limitation to flag for the class:** a reply can be
excellent and still score low here if it's well-written but phrased completely differently from the
reference — which is exactly why Section 5 adds model-based judgment on top.


In [4]:

from vertexai import types

computation_result = client.evals.evaluate(
    dataset=eval_dataset_v1,
    metrics=[
        types.Metric(name="rouge_1"),
        types.Metric(name="bleu"),
    ],
)
computation_result.show()


Computing Metrics for Evaluation Dataset: 100%|██████████| 16/16 [00:02<00:00,  7.14it/s]



## 5. Step 2 — Model-Based Metrics (LLM-as-Judge)

**What these measure:** rather than counting overlapping words, a **judge model** (Gemini, used
internally by the evaluation service) reads the prompt, the reference, and the candidate response,
then scores it against a defined rubric — much closer to how a human reviewer would actually assess
quality. We use two built-in **rubric metrics**:

- **`TEXT_QUALITY`** — overall coherence, grammar, and clarity.
- **`INSTRUCTION_FOLLOWING`** — whether the response actually addresses what was asked, not just
  whether it reads well.


In [7]:
import vertexai
vertexai.init(project=PROJECT_ID, location=REGION)

from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

eval_task = EvalTask(
    dataset=eval_data,
    metrics=[
        MetricPromptTemplateExamples.Pointwise.TEXT_QUALITY,
        MetricPromptTemplateExamples.Pointwise.INSTRUCTION_FOLLOWING,
    ],
    experiment="genai-eval-lab-experiment",
)
result = eval_task.evaluate(model="gemini-2.5-flash")
result.summary_metrics

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'gemini-2.5-flash'}
INFO:vertexai.evaluation._evaluation:Generating a total of 8 responses from Gemini model gemini-2.5-flash using genai module.
100%|██████████| 8/8 [00:20<00:00,  2.53s/it]
INFO:vertexai.evaluation._evaluation:All 8 responses are successfully generated from Gemini model gemini-2.5-flash using genai module.
INFO:vertexai.evaluation._evaluation:Multithreaded Batch Inference took: 20.30490461799991 seconds.
INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 16 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 16/16 [00:27<00:00,  1.70s/it]
INFO:vertexai.evaluation._evaluation:All 16 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:27.23645985400003 seconds


{'row_count': 8,
 'text_quality/mean': np.float64(5.0),
 'text_quality/std': 0.0,
 'instruction_following/mean': np.float64(4.875),
 'instruction_following/std': 0.3535533905932738}


> 🖥️ **Check it in the console:** **Agent Platform > Models > Evaluation** → this run appears
> listed with its summary scores — click in to see the same per-example breakdown as
> `model_based_result.show()` above, without needing the notebook open.



## 6. Comparing Two Models Side by Side (a Model Migration Decision)

**The real-world question this answers:** "Can we switch from `gemini-2.5-pro` to the cheaper,
faster `gemini-2.5-flash-lite` for this task without a meaningful quality drop?" — this is the
exact evaluation-driven decision Agent Platform's docs describe as **model migration**.


In [8]:

inference_pro = client.evals.run_inference(model="gemini-2.5-pro", src=eval_data)
inference_flash_lite = client.evals.run_inference(model="gemini-2.5-flash-lite", src=eval_data)

comparison_result = client.evals.evaluate(
    dataset=[inference_pro, inference_flash_lite],
    metrics=[
        types.RubricMetric.TEXT_QUALITY,
        types.RubricMetric.INSTRUCTION_FOLLOWING,
    ],
)
comparison_result.show()


Computing Metrics for Evaluation Dataset: 100%|██████████| 32/32 [01:41<00:00,  3.19s/it]



**How to read this:** the results show both candidates scored side by side on identical criteria.
If `gemini-2.5-flash-lite` scores within a small margin of `gemini-2.5-pro` on both metrics, that's
quantitative evidence supporting a migration to the cheaper model for this specific task — turning
a subjective "seems fine to me" into a defensible, reviewable decision.



## 7. Defining a Custom Metric

**Why built-in metrics aren't always enough:** `TEXT_QUALITY` and `INSTRUCTION_FOLLOWING` are
generic. This support team has one **business-specific rule**: every reply that acknowledges a
customer complaint should include a genuine **apology or acknowledgement**, not just a solution.
That's not a metric Google ships — so we define our own **rubric-based `PointwiseMetric`**, a
custom prompt template the same judge model uses to score each response against our specific
criterion.


In [11]:
import vertexai
vertexai.init(project=PROJECT_ID, location=REGION)

from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate
from google import genai

import time
from google.genai.errors import ClientError

def generate_with_retry(client, model, contents, max_retries=5):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=model, contents=contents)
        except ClientError as e:
            if e.code == 429 and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

acknowledgement_prompt_template = PointwiseMetricPromptTemplate(
    criteria={
        "acknowledges_customer": (
            "The response should open with or clearly include a genuine acknowledgement of the "
            "customer's frustration or situation, not just jump straight to a solution."
        )
    },
    rating_rubric={
        "1": "Does not acknowledge the customer's situation or feelings at all.",
        "3": "Partially acknowledges the situation, but feels perfunctory or rushed.",
        "5": "Clearly and genuinely acknowledges the customer's situation before addressing it.",
    },
)

acknowledgement_metric = PointwiseMetric(
    metric="acknowledges_customer",
    metric_prompt_template=acknowledgement_prompt_template,
)

# Generate responses directly (rather than reusing eval_dataset_v1, which is a newer-API object
# type that the legacy EvalTask framework doesn't consume directly) into a plain DataFrame.
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)
responses = []
for prompt in eval_data["prompt"]:
    r = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    responses.append(r.text)

responses_df = eval_data.assign(response=responses)

custom_metric_task = EvalTask(
    dataset=responses_df,
    metrics=[acknowledgement_metric],
    experiment="genai-eval-lab-experiment",
)
custom_metric_result = custom_metric_task.evaluate()
custom_metric_result.summary_metrics

INFO:vertexai.evaluation.metrics.metric_prompt_template:The `input_variables` parameter is empty. Only the `response` column is used for computing this model-based metric.


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 8 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 8/8 [00:20<00:00,  2.53s/it]
INFO:vertexai.evaluation._evaluation:All 8 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:20.247992533000343 seconds


{'row_count': 8,
 'acknowledges_customer/mean': np.float64(4.5),
 'acknowledges_customer/std': 1.4142135623730951}


**What to look for:** compare per-example scores here against the raw responses in Section 4.1 —
the two replies with an explicit "I'm sorry" / "I understand the frustration" opening should score
higher than the ones (like the restock question or the cancellation request) where an apology
wouldn't even make sense, which is a useful sanity check that the custom rubric is discriminating
correctly rather than scoring everything the same.



## 8. Prompt Regression Testing

**The real-world question this answers:** "We're about to change our production system prompt —
will this make responses better or worse?" This is where evaluation becomes a genuine **regression
test**, run the same way every time a prompt changes, not a one-off check.


In [12]:
from vertexai.evaluation import MetricPromptTemplateExamples

BASELINE_SYSTEM_PROMPT = "You are a customer support agent. Respond to the customer's message."

IMPROVED_SYSTEM_PROMPT = (
    "You are an empathetic customer support agent. Always acknowledge the customer's situation or "
    "frustration first, in your own words, before offering a solution. Keep responses concise — "
    "2-3 sentences."
)

def generate_with_system_prompt(system_prompt, df):
    genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)
    responses = []
    for prompt in df["prompt"]:
        r = generate_with_retry(genai_client, "gemini-2.5-flash", f"{system_prompt}\n\nCustomer message: {prompt}")
        responses.append(r.text)
    return df.assign(response=responses)

baseline_df = generate_with_system_prompt(BASELINE_SYSTEM_PROMPT, eval_data)
improved_df = generate_with_system_prompt(IMPROVED_SYSTEM_PROMPT, eval_data)

print("Sample baseline response:\n", baseline_df["response"].iloc[0])
print("\nSample improved-prompt response:\n", improved_df["response"].iloc[0])

  Rate limited, retrying in 1s...
  Rate limited, retrying in 1s...
  Rate limited, retrying in 2s...
Sample baseline response:
 Dear [Customer Name],

I'm so genuinely sorry to hear that your order arrived three days late and that the box was crushed. I can absolutely understand how frustrating and annoying that must have been.

While I'm relieved to know that the product itself is okay, that's certainly not the delivery experience we aim to provide, and we take full responsibility for the inconvenience caused by both the delay and the condition of the packaging. You're absolutely right to expect better.

We truly appreciate you bringing this to our attention. As a small token of our apology for the trouble, and to acknowledge the hassle you experienced, we'd like to offer you [e.g., a 15% discount on your next purchase / a $10 credit to your account]. We hope this helps make up for some of the frustration.

Thank you for your understanding and for continuing to choose us. Please don'

In [13]:
regression_task_baseline = EvalTask(
    dataset=baseline_df,
    metrics=[MetricPromptTemplateExamples.Pointwise.TEXT_QUALITY, acknowledgement_metric],
    experiment="genai-eval-lab-experiment",
)
baseline_result = regression_task_baseline.evaluate()

regression_task_improved = EvalTask(
    dataset=improved_df,
    metrics=[MetricPromptTemplateExamples.Pointwise.TEXT_QUALITY, acknowledgement_metric],
    experiment="genai-eval-lab-experiment",
)
improved_result = regression_task_improved.evaluate()

print("Baseline prompt scores:")
print(baseline_result.summary_metrics)
print("\nImproved prompt scores:")
print(improved_result.summary_metrics)

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 16 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 16/16 [00:27<00:00,  1.75s/it]
INFO:vertexai.evaluation._evaluation:All 16 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:27.940756512000007 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 16 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 16/16 [00:25<00:00,  1.57s/it]
INFO:vertexai.evaluation._evaluation:All 16 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:25.06513938299986 seconds


Baseline prompt scores:
{'row_count': 8, 'text_quality/mean': np.float64(5.0), 'text_quality/std': 0.0, 'acknowledges_customer/mean': np.float64(5.0), 'acknowledges_customer/std': 0.0}

Improved prompt scores:
{'row_count': 8, 'text_quality/mean': np.float64(5.0), 'text_quality/std': 0.0, 'acknowledges_customer/mean': np.float64(5.0), 'acknowledges_customer/std': 0.0}


**How to read this:** the `acknowledges_customer` custom metric should score noticeably higher for
the improved prompt (which explicitly instructs the model to acknowledge frustration first), while
`TEXT_QUALITY` should stay roughly comparable — evidence that the prompt change achieved its
specific intended goal without a general quality regression elsewhere. **This exact cell** — swap
in any new candidate prompt, re-run — is the reusable regression test the class should walk away
with: same dataset, same metrics, one line changed.

> 🖥️ **Check it in the console:** **Agent Platform > Models > Experiments** →
> `genai-eval-lab-experiment` — both runs from this section appear here alongside Section 5's and
> Section 7's runs, all comparable in one place.


## 9. Cleanup

**Good news:** this notebook created no continuously-billing resources — every call here (`evals`,
`genai.Client`) is a pay-per-request API call with no idle cost, unlike the deployed endpoints in
the companion MLOps and churn notebooks. There is nothing to delete.


In [14]:

print("No cleanup required — this notebook only made pay-per-call API requests")
print("(evaluation runs, generate_content calls) with no continuously-billing resources created.")


No cleanup required — this notebook only made pay-per-call API requests
(evaluation runs, generate_content calls) with no continuously-billing resources created.



> 🖥️ **Final console check:** **Agent Platform > Models > Evaluation** will retain a history of
> every evaluation run from this notebook — useful to keep as a record of past model/prompt
> comparisons, not something that needs cleaning up.
